# Chapter 5 — Exploratory Data Analysis and Data Quality
## 5.12 Practical Python EDA Workflow

**Source note:** This notebook follows the available Chapter 5 lesson content, reorganized for notebook study; it is not a word-for-word export of the conversation.

**Notebook use:** Read the explanations, run the code cells in order, interpret the results, and complete the exercises.

### 5.12.1 Reusable EDA workflow

Follow this sequence: problem definition → load/schema → quality audit → univariate analysis → relationships → target analysis → leakage audit → split and pipeline → baseline → findings. Establish a holdout before target-aware exploration.

### 5.12.2 Generate sample insurance data

In [ ]:
import numpy as np,pandas as pd
rng=np.random.default_rng(42);n=1500
df=pd.DataFrame({"customer_id":range(n),"age":rng.integers(18,81,n),"premium":rng.lognormal(7.4,.45,n),"tenure":rng.exponential(5,n),"policy_type":rng.choice(["Auto","Home","Life"],n)})
p=1/(1+np.exp(-(-2.4+.00015*(df.premium-1800)-.12*df.tenure)))
df["cancelled"]=rng.binomial(1,p)
df.loc[rng.choice(n,100,replace=False),"premium"]=np.nan

### 5.12.3 Inspect and audit

In [ ]:
print(df.shape,df.dtypes)
print(pd.DataFrame({"missing_n":df.isna().sum(),"missing_pct":100*df.isna().mean()}))
print("duplicates",df.duplicated().sum())
print(df.describe(include="all"))

### 5.12.4 Protect evaluation

In [ ]:
from sklearn.model_selection import train_test_split
X=df.drop(columns=["customer_id","cancelled"]);y=df.cancelled
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)

### 5.12.5 Model baseline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score,average_precision_score
prep=ColumnTransformer([("num",Pipeline([("imp",SimpleImputer(strategy="median")),("scale",StandardScaler())]),["age","premium","tenure"]),("cat",OneHotEncoder(handle_unknown="ignore"),["policy_type"])])
model=Pipeline([("prep",prep),("clf",LogisticRegression(max_iter=1000))])
model.fit(Xtr,ytr);scores=model.predict_proba(Xte)[:,1]
print("ROC-AUC",roc_auc_score(yte,scores),"AP",average_precision_score(yte,scores))

### 5.12.6 Document decisions

For each finding, record the evidence, the consequence for modeling, and the next experiment. Do not tune the threshold on the test set.

### 5.12.7 Practice

Compare mean vs median imputation using cross-validation; write a one-page EDA findings report.

### Review and reflection

- What did this section establish?
- Which assumptions matter?
- How would you use the technique in an unfamiliar real-world dataset?
- What could go wrong if it is applied incorrectly?